#  Phil Job Applier Agent - Rachel Briskman & Sanila Chowdhury (Daedalus Devs)

## Team Report #1 (03/24) - Created a function that takes in user's resume and parse it to output the extracted info & uses Skyvern to see how the library works for filling in the fields of a job website

<img src="AI_AGENT_WORKFLOW.png" width="700">

### Installs and Imports

In [ ]:

%pip install chromadb
# %pip install "skyvern==1.0.24"
%pip install skyvern
# %playwright install --with-deps chromium
# %playwright install chromium
%pip install langchain langchain-huggingface
%pip install python-dotenv

print ("Done installing")

In [ ]:
#installing dependencies for browser-use (an alternative to skyvern)
%pip install browser-use playwright
%pip install -U langchain-google-genai
%pip install -U browser-use
# %playwright install chromium

### Import API keys from .env file
#### (Note: This is for local use. Colab imports keys differently)

In [3]:
import os
from dotenv import load_dotenv
import platform
import sys


#Sanila needs to set the working directory explicity (macOS) but it's not necessary on Windows or WSL.
#added this macro so we don't have to manually comment/uncommment os.chdir() every time.

IS_MAC = platform.system() == "Darwin"

if IS_MAC:
    os.chdir("/Users/sanilachowdhury/Desktop/ai_agents")
    print("Running on macOS")
else:
    print("Not running on macOS, current directory:", os.getcwd())


load_dotenv()


gemini_key = os.getenv("GEMINI_API_KEY")
skyvern_key = os.getenv("SKYVERN_API_KEY")
phil_key = os.getenv("PHIL-KEY")


os.environ["ENABLE_AUTH"] = "false"

# from skyvern.forge.sdk import SkyvernForge
from skyvern import Skyvern
import nest_asyncio
import asyncio

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

# 2. Map it to the environment variables Skyvern/LiteLLM looks for
os.environ["GEMINI_API_KEY"] = gemini_key
# os.environ["GOOGLE_API_KEY"] = gemini_key  # Some versions prefer this
# Optional: Force Skyvern to use a specific model
os.environ["LLM_KEY"] = "gemini/gemini-1.5-flash"
print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")
# print("Keys loaded and environment configured.")

INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.schemas
INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.tables
INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.types
INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.constraints
INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.defaults
INFO     [alembic.runtime.plugins] setup plugin alembic.autogenerate.comments


Running on macOS
Key check: FOUND


### A website we're feeding into our agent to fill in the field (this website is used for testing purpose)

In [ ]:
url = "https://demo.applitools.com/"

### Agent takes in the filepath where the user's resume is stored and we prompt the agent to extract the info in a JSON that easy for both LLM and humans to read

In [4]:
import asyncio
import nest_asyncio
import os
from google import genai
from IPython.display import display, Markdown

nest_asyncio.apply()
client_genai = genai.Client(api_key=gemini_key)

def parse_resume(file_path, prompt="Please parse this resume and return its contents in a clear, readable format. Use sections with headers (like Contact, Summary, Experience, Education, Skills) and plain text or bullet points — no JSON or code blocks."):
    """
    Uploads a PDF to Gemini and returns the model's analysis.
    Caches the result so Gemini is only called once per resume.
    """
    cache_path = file_path.replace(".pdf", "_cached.json")

    # delete old cache if it exists to force re-parse with new format
    # if os.path.exists(cache_path):
    #     os.remove(cache_path)
    #     print(f"Cleared old cache at {cache_path}.")

    if os.path.exists(cache_path):
        print(f"Using cached resume data from {cache_path}...")
        with open(cache_path, "r") as f:
            return f.read()

    # upload user's resume
    print(f"Uploading {file_path}...")
    uploaded_file = client_genai.files.upload(
        file=file_path,
        config={"display_name": "User_PDF"}
    )

    # feeds LLM the prompt and uploaded resume PDF
    response = client_genai.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=[uploaded_file, prompt]
    )

    # save to cache so we don't call Gemini again
    with open(cache_path, "w") as f:
        f.write(response.text)
    print(f"Resume parsed and cached to {cache_path}")

    return response.text

# call the function
result = parse_resume("content/jakes-resume.pdf")
display(Markdown(f"<div style='font-size: 14px'>{result}</div>"))

Using cached resume data from content/jakes-resume_cached.json...


<div style='font-size: 14px'>Here's a breakdown of the resume:

**Contact**
*   **Name:** Jake Ryan
*   **Phone:** 123-456-7890
*   **Email:** jake@su.edu
*   **LinkedIn:** linkedin.com/in/jake
*   **GitHub:** github.com/jake

**Education**

*   **Southwestern University**
    *   Bachelor of Arts in Computer Science, Minor in Business
    *   Georgetown, TX
    *   August 2018 - May 2021
*   **Blinn College**
    *   Associate's in Liberal Arts
    *   Bryan, TX
    *   August 2014 - May 2018

**Experience**

*   **Undergraduate Research Assistant**
    *   Texas A&M University
    *   College Station, TX
    *   June 2020 - Present
        *   Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems.
        *   Developed a full-stack web application using Flask, React, PostgreSQL, and Docker to analyze GitHub data.
        *   Explored ways to visualize GitHub collaboration in a classroom setting.
*   **Information Technology Support Specialist**
    *   Southwestern University
    *   Georgetown, TX
    *   September 2018 - Present
        *   Communicate with managers to set up campus computers used on campus.
        *   Assess and troubleshoot computer problems brought by students, faculty, and staff.
        *   Maintain upkeep of computers, classroom equipment, and 200 printers across campus.
*   **Artificial Intelligence Research Assistant**
    *   Southwestern University
    *   Georgetown, TX
    *   May 2019 - July 2019
        *   Explored methods to generate video game dungeons based off of *The Legend of Zelda*.
        *   Developed a game in Java to test the generated dungeons.
        *   Contributed 50K+ lines of code to an established codebase via Git.
        *   Conducted a human subject study to determine which video game dungeon generation technique is enjoyable.
        *   Wrote an 8-page paper and gave multiple presentations on-campus.
        *   Presented virtually to the World Conference on Computational Intelligence.

**Projects**

*   **Gitlytics**
    *   Python, Flask, React, PostgreSQL, Docker
    *   June 2020 - Present
        *   Developed a full-stack web application using Flask serving a REST API with React as the frontend.
        *   Implemented GitHub OAuth to get data from user's repositories.
        *   Visualized GitHub data to show collaboration.
        *   Used Celery and Redis for asynchronous tasks.
*   **Simple Paintball**
    *   Spigot API, Java, Maven, TravisCI, Git
    *   May 2018 - May 2020
        *   Developed a Minecraft server plugin to entertain kids during free time for a previous job.
        *   Published plugin to websites gaining 2K+ downloads and an average 4.5/5-star review.
        *   Implemented continuous delivery using TravisCI to build the plugin upon a new release.
        *   Collaborated with Minecraft server administrators to suggest features and get feedback about the plugin.

**Technical Skills**

*   **Languages:** Java, Python, C/C++, SQL (Postgres), JavaScript, HTML/CSS, R
*   **Frameworks:** React, Node.js, Flask, JUnit, WordPress, Material-UI, FastAPI
*   **Developer Tools:** Git, Docker, TravisCI, Google Cloud Platform, VS Code, Visual Studio, PyCharm, IntelliJ, Eclipse
*   **Libraries:** pandas, NumPy, Matplotlib</div>

## Team Report #2 (03/31) - Created human in the loop for resume info verification, set up ChromaDB, and opened a window of the job website (dummy url) and filled in a few fields.
### (Replaced Skyvern with Browser_Use). This code will not run on Colab, only locally.

<img src="AI_AGENT_WORKFLOW_REPORT_2.png" width="800">

### Developed a human in the loop where agent takes in the extracted info and ask the user if the info is correct. Until the user is satisifed, the agent will keep updating the extracted info based on user's feedback.

In [5]:
from IPython.display import display, Markdown

def verify_resume_info(extracted_info):
    current_info = extracted_info
    while True:
        print("\n" + "="*10)
        print("EXTRACTED RESUME INFO:")
        print("="*10)
        display(Markdown(f"<div style='font-size: 16px'>{current_info}</div>"))
        print("="*10)

        user_confirm = input("\nIs this information correct? (y/n): ").strip().lower()
        print(f"Is this information correct? (y/n): {user_confirm}")

        if user_confirm == "y":
            print("\nGreat! Moving forward with this information.")
            break
        else:
            correction = input("\nWhat needs to be corrected or added? Please describe: ").strip()
            print(f"What needs to be corrected or added? Please describe: {correction}")

            print("\nUpdating extracted info...")
            response = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"""Here is the currently extracted resume information:
{current_info}
The user wants the following correction or addition:
{correction}
Please update the extracted resume JSON accordingly and return the full updated JSON."""
                ]
            )
            current_info = response.text
            print("\nInfo updated! Let's review it again.")
    return current_info


# call the function to store the extracted info
result = parse_resume("content/jakes-resume.pdf")

# run verify_resume_info to verify and correct
final_info = verify_resume_info(result)

Using cached resume data from content/jakes-resume_cached.json...

EXTRACTED RESUME INFO:


<div style='font-size: 16px'>Here's a breakdown of the resume:

**Contact**
*   **Name:** Jake Ryan
*   **Phone:** 123-456-7890
*   **Email:** jake@su.edu
*   **LinkedIn:** linkedin.com/in/jake
*   **GitHub:** github.com/jake

**Education**

*   **Southwestern University**
    *   Bachelor of Arts in Computer Science, Minor in Business
    *   Georgetown, TX
    *   August 2018 - May 2021
*   **Blinn College**
    *   Associate's in Liberal Arts
    *   Bryan, TX
    *   August 2014 - May 2018

**Experience**

*   **Undergraduate Research Assistant**
    *   Texas A&M University
    *   College Station, TX
    *   June 2020 - Present
        *   Developed a REST API using FastAPI and PostgreSQL to store data from learning management systems.
        *   Developed a full-stack web application using Flask, React, PostgreSQL, and Docker to analyze GitHub data.
        *   Explored ways to visualize GitHub collaboration in a classroom setting.
*   **Information Technology Support Specialist**
    *   Southwestern University
    *   Georgetown, TX
    *   September 2018 - Present
        *   Communicate with managers to set up campus computers used on campus.
        *   Assess and troubleshoot computer problems brought by students, faculty, and staff.
        *   Maintain upkeep of computers, classroom equipment, and 200 printers across campus.
*   **Artificial Intelligence Research Assistant**
    *   Southwestern University
    *   Georgetown, TX
    *   May 2019 - July 2019
        *   Explored methods to generate video game dungeons based off of *The Legend of Zelda*.
        *   Developed a game in Java to test the generated dungeons.
        *   Contributed 50K+ lines of code to an established codebase via Git.
        *   Conducted a human subject study to determine which video game dungeon generation technique is enjoyable.
        *   Wrote an 8-page paper and gave multiple presentations on-campus.
        *   Presented virtually to the World Conference on Computational Intelligence.

**Projects**

*   **Gitlytics**
    *   Python, Flask, React, PostgreSQL, Docker
    *   June 2020 - Present
        *   Developed a full-stack web application using Flask serving a REST API with React as the frontend.
        *   Implemented GitHub OAuth to get data from user's repositories.
        *   Visualized GitHub data to show collaboration.
        *   Used Celery and Redis for asynchronous tasks.
*   **Simple Paintball**
    *   Spigot API, Java, Maven, TravisCI, Git
    *   May 2018 - May 2020
        *   Developed a Minecraft server plugin to entertain kids during free time for a previous job.
        *   Published plugin to websites gaining 2K+ downloads and an average 4.5/5-star review.
        *   Implemented continuous delivery using TravisCI to build the plugin upon a new release.
        *   Collaborated with Minecraft server administrators to suggest features and get feedback about the plugin.

**Technical Skills**

*   **Languages:** Java, Python, C/C++, SQL (Postgres), JavaScript, HTML/CSS, R
*   **Frameworks:** React, Node.js, Flask, JUnit, WordPress, Material-UI, FastAPI
*   **Developer Tools:** Git, Docker, TravisCI, Google Cloud Platform, VS Code, Visual Studio, PyCharm, IntelliJ, Eclipse
*   **Libraries:** pandas, NumPy, Matplotlib</div>

Is this information correct? (y/n): y

Great! Moving forward with this information.


### Setting up ChromaDB which will store the correct parsed info of user's resume

In [6]:
import chromadb
import os
import hashlib
from dotenv import load_dotenv

# debug - check if values are loaded
print(f"CHROMA_HOST: {os.getenv('CHROMA_HOST')}")
print(f"CHROMA_TENANT: {os.getenv('CHROMA_TENANT')}")
print(f"CHROMA_DATABASE: {os.getenv('CHROMA_DATABASE')}")
print(f"CHROMA_API_KEY: {'FOUND' if os.getenv('CHROMA_API_KEY') else 'MISSING'}")

chroma_client = chromadb.HttpClient(
    ssl=True,
    host=os.getenv("CHROMA_HOST"),
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE"),
    headers={"x-chroma-token": os.getenv("CHROMA_API_KEY")}
)
collection = chroma_client.get_or_create_collection(name="resumes")
print(f"Connected to ChromaDB! Total resumes stored: {collection.count()}")

def store_resume_in_chromadb(resume_text, file_path):
    doc_id = hashlib.md5(file_path.encode()).hexdigest()
    collection.upsert(
        documents=[resume_text],
        ids=[doc_id],
        metadatas=[{"file_path": file_path}]
    )
    print(f"Resume stored! ID: {doc_id}")
    return doc_id

# store the verified resume
doc_id = store_resume_in_chromadb(final_info, "content/jakes-resume.pdf")
print("Resume stored successfully!")

CHROMA_HOST: api.trychroma.com
CHROMA_TENANT: 7eab8b90-a03c-4bc3-9704-e5e4f8a3f82b
CHROMA_DATABASE: phil-job-agent
CHROMA_API_KEY: FOUND
Connected to ChromaDB! Total resumes stored: 1
Resume stored! ID: b7a0358df030937c049bb9754117f552
Resume stored successfully!


## Team Report #3 (04/14) - TBD

### A function to write to a CSV file with details of the job application. If a CSV file does not exist, it will create one.

In [5]:
import csv
import os

def save_application_history(url, company_name, time_applied, job_title):
    file_exists = os.path.isfile("application_history.csv")
    with open("application_history.csv", mode="a", newline="") as csvfile:
        fieldnames = ["url", "company_name", "time_applied", "job_title"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        if not file_exists:
            writer.writeheader()

        writer.writerow({"url": url, "company_name": company_name, "time_applied": time_applied, "job_title": job_title})
    print("Application history updated.")


### Created a popup window of the job website that user will feed into Phil

In [ ]:
import asyncio
import os
from browser_use import Agent, Browser, BrowserProfile
from browser_use.llm import ChatOpenAI  # ← browser-use's own wrapper, NOT langchain_openai

# --- Configuration ---
OPENROUTER_API_KEY = os.getenv("OPENROUTER_KEY")
TARGET_URL = "https://demo.applitools.com/"
MODEL = "anthropic/claude-3.5-sonnet"
MODEL = "anthropic/claude-sonnet-4.5"
MODEL = "google/gemini-2.5-flash-lite"

async def main():
    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        model=MODEL,
    )

    browser = Browser(
        browser_profile=BrowserProfile(
            headless=False,
            disable_security=False,
            keep_alive=True,
        )
    )

    task = (
        f"Go to {TARGET_URL}. "
        f"Type 'test_user' into the username field. "
        f"Type 'password123' into the password field. "
        f"STOP. Do NOT click any button. Do NOT press Enter. Do NOT submit the form. "
        f"Return DONE when both fields are filled."
    )

    agent = Agent(
        task=task,
        llm=llm,
        browser=browser,
        use_vision=True,
        max_actions=10,
    )

    result = await agent.run()
    print("Result:", result)

    # await browser.close()


# Jupyter-compatible (no asyncio.run)
await main()

INFO     [service] Using anonymized telemetry, see https://docs.browser-use.com/development/monitoring/telemetry.
INFO     [Agent] 🔗 Found URL in task: https://demo.applitools.com/, adding as initial action...
INFO     [Agent] 🎯 Task: Go to https://demo.applitools.com/. Type 'test_user' into the username field. Type 'password123' into the password field. STOP. Do NOT click any button. Do NOT press Enter. Do NOT submit the form. Return DONE when both fields are filled.
INFO     [Agent] Starting a browser-use agent with version 0.12.6, with provider=openai and model=google/gemini-2.5-flash-lite
INFO     [Agent]   ▶️   navigate: url: https://demo.applitools.com/, new_tab: False
WARNING  [BrowserSession] ⚠️ Page readiness timeout (8.0s, 8290ms) for https://demo.applitools.com/
INFO     [tools] 🔗 Navigated to https://demo.applitools.com/
INFO     [Agent] 

INFO     [Agent] 📍 Step 1:
INFO     [Agent]   👍 Eval: Successfully navigated to the demo.applitools.com page. Verdict: Success
INFO     

Result: AgentHistoryList(all_results=[ActionResult(is_done=False, success=None, judgement=None, error=None, attachments=None, images=None, long_term_memory='Found initial url and automatically loaded it. Navigated to https://demo.applitools.com/', extracted_content='🔗 Navigated to https://demo.applitools.com/', include_extracted_content_only_once=False, metadata=None, include_in_memory=False), ActionResult(is_done=False, success=None, judgement=None, error=None, attachments=None, images=None, long_term_memory="Typed 'test_user'", extracted_content="Typed 'test_user'", include_extracted_content_only_once=False, metadata={'input_x': 756.0, 'input_y': 407.1875}, include_in_memory=False), ActionResult(is_done=False, success=None, judgement=None, error='1 validation error for AgentOutput\n  Invalid JSON: EOF while parsing a string at line 2 column 351 [type=json_invalid, input_value=\'{\\n  "thinking": "The us...g the password field, I\', input_type=str]\n    For further information visit h

WARNING  [BrowserSession] [SessionManager] Agent focus target 549ED8A8... detached! Current focus: None (already cleared). Auto-recovering by switching to another target...
WARNING  [BrowserSession] [SessionManager] No tabs remain! Creating new tab for agent...
WARNING  [BrowserSession] Cannot navigate - browser not connected
INFO     [BrowserSession] [SessionManager] Created new tab 43C8A868... for agent
INFO     [BrowserSession] [SessionManager] ✅ Agent focus recovered: 43C8A868...
WARNING  [BrowserSession] [SessionManager] Agent focus target 43C8A868... detached! Current focus: None (already cleared). Auto-recovering by switching to another target...
WARNING  [BrowserSession] [SessionManager] No tabs remain! Creating new tab for agent...
WARNING  [BrowserSession] Cannot navigate - browser not connected
ERROR    [BrowserSession] [SessionManager] ❌ Error during agent_focus recovery: RuntimeError: {'code': -32000, 'message': 'Failed to open a new tab'}
WARNING  [BrowserSession] 🔌 CDP W

### Phil fills in the application field

In [ ]:
import json
from datetime import datetime
from browser_use import Agent, BrowserProfile
from browser_use.browser import BrowserSession
from browser_use.llm import ChatOpenAI

# Opens the job application in a new browser window.
# Waits for user to confirm they are signed in before filling in the fields.
async def fill_application(job_url: str, resume_text: str):
    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_KEY"),
        model="openai/gpt-4o",
    )

    browser_session = BrowserSession(
        browser_profile=BrowserProfile(headless=False, keep_alive=True)
    )

    # Open the browser to the job URL first
    agent_open = Agent(
        task=f"Go to {job_url} and wait. Do not fill anything.",
        llm=llm,
        browser_session=browser_session,
        max_steps=3,
    )
    await agent_open.run()

    # Wait for user to confirm they are signed in
    signed_in = input("Are you signed in (if needed)? (y/n): ").strip().lower()
    while signed_in != "y":
        signed_in = input("Please sign in and then enter y to continue: ").strip().lower()

    # Now fill in the fields
    agent_fill = Agent(
    task=(
        f"The page {job_url} is already open. "
        f"Use this resume to fill out the application:\n{resume_text}\n\n"
        "Instructions:\n"
        "1. Find every empty field on the page.\n"
        "2. If the field can be answered from the resume, fill it in.\n"
        "3. For DROPDOWN fields: click the dropdown and select the most appropriate option based on the resume.\n"
        "4. For SENSITIVE fields (Gender, Race, Veteran status, Disability status, Ethnicity):\n"
        "   - Open the dropdown, pick the FIRST available option and immediately move on.\n"
        "   - Do NOT re-open or reconsider. Just select and move to the next field.\n"
        "5. For SPONSORSHIP/WORK AUTHORIZATION fields:\n"
        "   - If the resume indicates the candidate is authorized to work in the country, select 'Yes' for authorization and 'No' for sponsorship.\n"
        "   - If the resume indicates the candidate is NOT authorized, select 'Yes' for sponsorship needed.\n"
        "   - If unclear from the resume, select 'Yes' for authorization and 'No' for sponsorship as the default.\n"
        "6. For COUNTRY fields: select 'United States' unless the resume says otherwise.\n"
        "7. For FILE UPLOAD fields (resume, CV): upload the file located at 'content/jakes-resume.pdf'.\n"
        "8. For other FILE UPLOAD fields (cover letter, portfolio, other documents): skip them.\n"
        "9. Do NOT click submit or login.\n"
        "10. When all fields are processed, return ONLY a JSON like this:\n"
        '{"filled": ["field1", "field2"], "empty": ["field3"], "skipped_uploads": ["cover letter"], "company_name": "Google", "job_title": "Software Engineer"}'
    ),
    llm=llm,
    browser_session=browser_session,
    use_vision=True,
    max_steps=50,
    max_actions_per_step=5,
    include_attributes=["id", "name", "type", "placeholder", "aria-label", "label", "role", "data-testid"],
)

    result = await agent_fill.run()
    final = result.final_result()
    print("Agent result:", final)

    try:
        parsed = json.loads(final)
        empty_fields = parsed.get("empty", [])
        company_name = parsed.get("company_name", "Unknown")
        job_title = parsed.get("job_title", "Unknown")

        if empty_fields:
            print(f"⚠️ Empty fields remaining: {empty_fields}")
        else:
            print("✅ All fields filled!")
            save_application_history(
                url=job_url,
                company_name=company_name,
                time_applied=datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
                job_title=job_title,
            )
    except Exception:
        if final and "empty" not in final.lower():
            save_application_history(
                url=job_url,
                company_name="Unknown",
                time_applied=datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
                job_title="Unknown",
            )
        else:
            print("⚠️ Could not confirm all fields filled.")

    return final

### Agent will first ask user if they would like to upload a new resume if it's their first time. Otherwise, it will grab the resume that's stored in ChromaDB and use that to fill in the field

In [ ]:
async def apply_to_job_with_agent():
    resume_exists = input("Would you like to upload a new resume? Enter y if this is your first time (y/n): ")
    
    while(resume_exists != "y" and resume_exists != "n"):
        resume_exists = input("Invalid input. Please enter y or n:")
    
    if resume_exists == "y":
        file_path = input("Please enter the path to your resume PDF: ").strip()
        extracted_info = parse_resume(file_path)
        verified_info = verify_resume_info(extracted_info)
        store_resume_in_chromadb(verified_info, file_path)
        resume_text = verified_info
    else:
        print("Using existing resume data. Moving on to job application...")
        results = collection.get(limit=1, include=["documents"])
        if not results["documents"]:
            raise RuntimeError("No resume found in ChromaDB. Please upload a resume first.")
        resume_text = results["documents"][0]
        print("✅ Resume fetched from ChromaDB")

    url = input("Please enter the URL of the job application you are applying to: ").strip()
    await fill_application(url, resume_text)

await apply_to_job_with_agent()


Using existing resume data. Moving on to job application...
✅ Resume fetched from ChromaDB


INFO     [service] Using anonymized telemetry, see https://docs.browser-use.com/development/monitoring/telemetry.
INFO     [Agent] 🔗 Found URL in task: https://job-boards.greenhouse.io/attentive/jobs/4209296009, adding as initial action...
INFO     [Agent] 🎯 Task: Go to https://job-boards.greenhouse.io/attentive/jobs/4209296009 and wait. Do not fill anything.
INFO     [Agent] Starting a browser-use agent with version 0.12.6, with provider=openai and model=openai/gpt-4o
INFO     [Agent]   ▶️   navigate: url: https://job-boards.greenhouse.io/attentive/jobs/4209296009, new_tab: False
INFO     [tools] 🔗 Navigated to https://job-boards.greenhouse.io/attentive/jobs/4209296009
INFO     [Agent] 

INFO     [Agent] 📍 Step 1:
INFO     [Agent]   👍 Eval: Successfully navigated to the specified URL. Verdict: Success
INFO     [Agent]   🧠 Memory: Navigated to the job application page for AI Intern at Attentive. Waiting as per user request.
INFO     [Agent]   🎯 Next goal: Wait on the current page as in

Agent result: The application form was filled with available information from Jake Ryan's resume. The following fields were completed: first name, last name, email, phone number, country, gender, race, and veteran status. The resume upload was skipped due to a missing file path. Here's the final JSON documentation:

{"filled": ["first name", "last name", "email", "phone number", "country", "gender", "race", "veteran status"], "empty": ["LinkedIn Profile", "Website"], "skipped_uploads": ["resume", "cover letter"], "company_name": "Attentive", "job_title": "AI Intern"}

Attachments:

application_status.json:
{"filled": ["first name", "last name", "email", "phone number", "country", "gender", "race", "veteran status"], "empty": ["LinkedIn Profile", "Website"], "skipped_uploads": ["resume", "cover letter"], "company_name": "Attentive", "job_title": "AI Intern"}

⚠️ Could not confirm all fields filled.


WARNING  [BrowserSession] [SessionManager] Agent focus target 13FFAD86... detached! Current focus: None (already cleared). Auto-recovering by switching to another target...
WARNING  [BrowserSession] [SessionManager] No tabs remain! Creating new tab for agent...
WARNING  [BrowserSession] Cannot navigate - browser not connected
INFO     [BrowserSession] [SessionManager] Created new tab A0226828... for agent
INFO     [BrowserSession] [SessionManager] ✅ Agent focus recovered: A0226828...
WARNING  [BrowserSession] [SessionManager] Agent focus target A0226828... detached! Current focus: None (already cleared). Auto-recovering by switching to another target...
WARNING  [BrowserSession] [SessionManager] No tabs remain! Creating new tab for agent...
WARNING  [BrowserSession] Cannot navigate - browser not connected
INFO     [BrowserSession] [SessionManager] Created new tab 3B47B5C1... for agent
INFO     [BrowserSession] [SessionManager] ✅ Agent focus recovered: 3B47B5C1...
WARNING  [BrowserSessi

### Agent handling short-answer questions field

## PLANS FOR NEXT MEETING:


## Plans for spring break:
* Implementing a function where our agent to fill in the fields from the job website using Skyvern (we have started this process but currently experiencing errors)
* Implementing a loop for our agent to handle when it comes to short-answer response
  * Have a prompt that will tell the LLM to generate answer based on the context given and the info it extracted from user's resume
  * Have a loop where as long as the answer is too vague, keep asking user for more info and implement a function for LLM to do web search (depending on the short answer question)
  * Have LLM generate a short answer response and fill out the field with Skyvern
* IF WE FIND TIME -> Have an evalulation where user can give the LLM feedback at the end of the overall performance and store that in ChromaDB to help Phil improve